# Securelytics: train all 4 fraud models on Google Colab

Trains **HistGradientBoosting, XGBoost, LightGBM and CatBoost** on the Sparkov dataset, compares them, and gives you a zip with the trained models to put in your project folder. Nothing is installed on your laptop.

**Before you start (one time):**
1. In Google Drive, create the folder `MyDrive/Securelytics/`.
2. Upload your project's **`src`** folder into it, so you have `MyDrive/Securelytics/src/securelytics/...` (it's small, about 100 KB).
3. *Optional:* upload `fraudTrain.csv` and `fraudTest.csv` to `MyDrive/Securelytics/data/raw/`. If you skip this, the notebook downloads them from Kaggle.

Then run every cell in order (**Runtime → Run all**). The whole thing takes about 20–30 minutes on the free CPU runtime; no GPU needed.

In [ ]:
# 1. Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/Securelytics')
WORK = pathlib.Path('/content/work')          # fast local disk for the actual work
assert (DRIVE / 'src' / 'securelytics' / 'train.py').is_file(), \
    f'Upload your project src folder to {DRIVE}/src first (see instructions above)'
print('Found project code in', DRIVE / 'src')

In [ ]:
# 2. Install the model libraries (Colab already has most of the rest)
!pip -q install xgboost lightgbm catboost kagglehub
import sklearn, xgboost, lightgbm, catboost, numpy, pandas
print('scikit-learn', sklearn.__version__, '| xgboost', xgboost.__version__,
      '| lightgbm', lightgbm.__version__, '| catboost', catboost.__version__)

In [ ]:
# 3. Copy code and data to the local disk
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(DRIVE / 'src', WORK / 'src')
raw = WORK / 'data' / 'raw'
raw.mkdir(parents=True)

drive_raw = DRIVE / 'data' / 'raw'
if (drive_raw / 'fraudTrain.csv').is_file() and (drive_raw / 'fraudTest.csv').is_file():
    print('Using CSVs from Google Drive')
    for f in ['fraudTrain.csv', 'fraudTest.csv']:
        shutil.copy(drive_raw / f, raw / f)
else:
    print('CSVs not on Drive: downloading the dataset from Kaggle...')
    import kagglehub
    path = pathlib.Path(kagglehub.dataset_download('kartik2112/fraud-detection'))
    for f in ['fraudTrain.csv', 'fraudTest.csv']:
        src = next(path.rglob(f))
        shutil.copy(src, raw / f)
print(sorted(p.name for p in raw.iterdir()))

os.environ['SECURELYTICS_ROOT'] = str(WORK)
os.environ['PYTHONPATH'] = str(WORK / 'src')
os.chdir(WORK)

## Step 1: clean the data and build the real-time features (~2 min)

In [ ]:
!python -m securelytics.prepare

## Step 2: train and compare all four models (~15–25 min)

In [ ]:
!python -m securelytics.train

## Step 3: EDA charts (~1 min)

In [ ]:
!python -m securelytics.eda

## Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((WORK / 'reports' / 'model_comparison.md').read_text()))
display(Image(str(WORK / 'reports' / 'model_comparison_curves.png')))
display(Image(str(WORK / 'reports' / 'model_comparison_bars.png')))

## Step 4: save the results and download them

This writes `requirements-models.txt` with the exact library versions used here, so the Docker image on your laptop loads the models with the same versions. The zip is also saved to `MyDrive/Securelytics/colab_output/`.

In [ ]:
import platform, importlib.metadata as md_
# only the libraries whose pickles must match exactly (numpy/scipy stay as in requirements.txt)
pkgs = ['scikit-learn', 'joblib', 'xgboost', 'lightgbm', 'catboost']
lock = ['# Library versions used to train the models in models/ (generated by train_on_colab.ipynb)',
        f'# Python {platform.python_version()}']
lock += [f'{p}=={md_.version(p)}' for p in pkgs]
(WORK / 'requirements-models.txt').write_text('\n'.join(lock) + '\n')
print('\n'.join(lock))

out = WORK / 'colab_output'
if out.exists():
    shutil.rmtree(out)
out.mkdir()
shutil.copytree(WORK / 'models', out / 'models')
shutil.copytree(WORK / 'reports', out / 'reports')
shutil.copy(WORK / 'requirements-models.txt', out / 'requirements-models.txt')
zip_path = shutil.make_archive('/content/securelytics_models', 'zip', out)

(DRIVE / 'colab_output').mkdir(exist_ok=True)
shutil.copy(zip_path, DRIVE / 'colab_output' / 'securelytics_models.zip')
print('Saved to Drive:', DRIVE / 'colab_output' / 'securelytics_models.zip')

from google.colab import files
files.download(zip_path)

## Back on your laptop

1. Unzip `securelytics_models.zip` **into `E:\\Securelytics`** and say *Yes* to replacing files. You get:
   - `models\\` (4 models + `registry.json` + `card_state_snapshot.json`)
   - `reports\\` (comparison table and charts)
   - `requirements-models.txt` (in the project root)
2. Rebuild and start: `docker compose build` then `docker compose up`.
3. Open http://localhost:8501 → **Admin panel** to switch between the four models.